# GIVP — Benchmark Literature Comparison

Empirical comparison of **GIVP (full pipeline)** vs. a **GRASP-only baseline** on four
standard continuous optimisation benchmarks over **30 independent runs** per configuration.

## Purpose

| Goal | Detail |
|------|--------|
| Reproducibility | Each run uses an explicit `seed` via `GIVPConfig`, giving byte-for-byte identical results. |
| Scientific comparison | The GRASP-only baseline mirrors the plain GRASP of Feo & Resende (1995) by disabling ILS, VND, elite pool, and convergence monitor. |
| Export | Results are serialised to `benchmark_literature_comparison_results.json` in this directory. |

## References
- Feo, T.A. & Resende, M.G.C. (1995). *Greedy randomized adaptive search procedures.* Journal of Global Optimization, 6, 109–133.
- Festa, P. & Resende, M.G.C. (2011). *GRASP: An annotated bibliography.* Essays and Surveys in Metaheuristics. Springer.
- Molina, D. et al. (2020). *An Insight into Bio-inspired and Evolutionary Algorithms for Global Optimization.* IEEE Access.

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
from __future__ import annotations

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from givp import GIVPConfig, givp
from givp import benchmarks as bm

print("givp imported successfully")

## 1. Benchmark functions

All four functions have a **known global minimum of 0** at the canonical point.
We use `n = 10` dimensions — a standard dimensionality in GRASP / metaheuristic
comparison papers.

| Function | Global min | Bounds |
|----------|-----------|--------|
| Sphere | 0 at **0** | [-5.12, 5.12]^10 |
| Rosenbrock | 0 at **1** | [-5, 10]^10 |
| Rastrigin | 0 at **0** | [-5.12, 5.12]^10 |
| Ackley | 0 at **0** | [-32.768, 32.768]^10 |

In [ ]:
# ── Benchmark definitions ──────────────────────────────────────────────────
N_DIMS = 10

BENCHMARKS: dict[str, dict] = {
    "Sphere": {
        "func": bm.sphere,
        "bounds": [(-5.12, 5.12)] * N_DIMS,
        "optimum": 0.0,
    },
    "Rosenbrock": {
        "func": bm.rosenbrock,
        "bounds": [(-5.0, 10.0)] * N_DIMS,
        "optimum": 0.0,
    },
    "Rastrigin": {
        "func": bm.rastrigin,
        "bounds": [(-5.12, 5.12)] * N_DIMS,
        "optimum": 0.0,
    },
    "Ackley": {
        "func": bm.ackley,
        "bounds": [(-32.768, 32.768)] * N_DIMS,
        "optimum": 0.0,
    },
}

N_RUNS = 30
SEEDS = list(range(N_RUNS))  # seeds 0..29 — fully deterministic

print(f"Benchmarks: {list(BENCHMARKS)}")
print(f"Runs per config: {N_RUNS}")

## 2. Algorithm configurations

| Config | Description |
|--------|-------------|
| **GIVP-full** | Complete GRASP-ILS-VND-PR pipeline with adaptive α, elite pool, convergence monitor, and path relinking. |
| **GRASP-only** | Construction + trivial local search only. ILS, VND, elite pool, path relinking, and convergence monitor are all disabled. This mirrors the basic GRASP of Feo & Resende (1995). |

In [ ]:
# ── Algorithm configurations ───────────────────────────────────────────────

def make_givp_full(seed: int) -> GIVPConfig:
    """Full GIVP pipeline — ILS + VND + Path Relinking + elite pool."""
    return GIVPConfig(
        max_iterations=200,
        alpha=0.12,
        adaptive_alpha=True,
        alpha_min=0.08,
        alpha_max=0.18,
        vnd_iterations=200,
        ils_iterations=10,
        perturbation_strength=4,
        use_elite_pool=True,
        elite_size=7,
        path_relink_frequency=8,
        use_cache=True,
        cache_size=10_000,
        early_stop_threshold=80,
        use_convergence_monitor=True,
        time_limit=30.0,
        # seed passed via api.py _set_seed wrapper
    )


def make_grasp_only(seed: int) -> GIVPConfig:
    """GRASP-only baseline — construction + minimal local search, no ILS/VND/PR."""
    return GIVPConfig(
        max_iterations=200,
        alpha=0.12,
        adaptive_alpha=False,
        vnd_iterations=1,         # effectively disabled
        ils_iterations=1,         # effectively disabled
        perturbation_strength=0,
        use_elite_pool=False,
        use_convergence_monitor=False,
        use_cache=True,
        cache_size=10_000,
        early_stop_threshold=200,  # do not stop early
        time_limit=30.0,
    )


CONFIGS = {
    "GIVP-full": make_givp_full,
    "GRASP-only": make_grasp_only,
}

print("Configurations:", list(CONFIGS))

## 3. Run experiments

`seed` is passed to `givp()` which internally calls `_set_seed` via a `ContextVar`,
ensuring deterministic `SeedSequence` spawning — byte-identical results every run.

In [ ]:
# ── Experiment runner ──────────────────────────────────────────────────────
records: list[dict] = []

total = N_RUNS * len(BENCHMARKS) * len(CONFIGS)
done = 0

for algo_name, cfg_factory in CONFIGS.items():
    for fn_name, spec in BENCHMARKS.items():
        func = spec["func"]
        bounds = spec["bounds"]
        for seed in SEEDS:
            cfg = cfg_factory(seed)
            t0 = time.perf_counter()
            result = givp(func, bounds, config=cfg, seed=seed)
            elapsed = time.perf_counter() - t0
            records.append(
                {
                    "algorithm": algo_name,
                    "function": fn_name,
                    "seed": seed,
                    "fun": result.fun,
                    "nit": result.nit,
                    "nfev": result.nfev,
                    "time_s": elapsed,
                    "success": result.success,
                }
            )
            done += 1
            if done % 40 == 0 or done == total:
                print(f"  [{done:3d}/{total}] {algo_name:12s} | {fn_name:12s} | seed={seed}")

print(f"\nDone. {len(records)} records collected.")

## 4. Summary statistics

In [ ]:
# ── Summary table ──────────────────────────────────────────────────────────
df = pd.DataFrame(records)

summary = (
    df.groupby(["function", "algorithm"])["fun"]
    .agg(
        mean="mean",
        std="std",
        best="min",
        median="median",
        worst="max",
    )
    .reset_index()
)

# Merge NFev and time
nfev_time = (
    df.groupby(["function", "algorithm"])[["nfev", "time_s"]]
    .mean()
    .round({"nfev": 0, "time_s": 3})
    .reset_index()
)
summary = summary.merge(nfev_time, on=["function", "algorithm"])

# Format for display
disp = summary.copy()
disp["mean ± std"] = disp.apply(
    lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1
)
disp = disp[["function", "algorithm", "mean ± std", "best", "median", "nfev", "time_s"]]
disp.columns = ["Function", "Algorithm", "Mean ± Std", "Best", "Median", "NFev (mean)", "Time (s)"]

print(disp.to_string(index=False))

## 5. Visualisations

In [ ]:
# ── Boxplot: objective value per function × algorithm ──────────────────────
fn_names = list(BENCHMARKS)
algo_names = list(CONFIGS)
fig, axes = plt.subplots(1, len(fn_names), figsize=(4 * len(fn_names), 5), sharey=False)

palette = {"GIVP-full": "#2196F3", "GRASP-only": "#FF9800"}

for ax, fn_name in zip(axes, fn_names):
    data = [
        df[(df["function"] == fn_name) & (df["algorithm"] == a)]["fun"].values
        for a in algo_names
    ]
    bp = ax.boxplot(
        data,
        labels=algo_names,
        patch_artist=True,
        medianprops=dict(color="black", linewidth=2),
    )
    for patch, a in zip(bp["boxes"], algo_names):
        patch.set_facecolor(palette[a])
        patch.set_alpha(0.7)
    ax.set_title(fn_name, fontsize=13, fontweight="bold")
    ax.set_ylabel("Objective value", fontsize=10)
    ax.yaxis.set_major_formatter(mticker.ScalarFormatter(useMathText=True))
    ax.grid(axis="y", linestyle="--", alpha=0.4)

fig.suptitle(
    f"GIVP-full vs GRASP-only — {N_RUNS} runs × n={N_DIMS}",
    fontsize=14,
    fontweight="bold",
    y=1.02,
)
plt.tight_layout()
plt.savefig(
    Path(__file__).parent / "benchmark_boxplot.png" if "__file__" in dir() else "benchmark_boxplot.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

In [ ]:
# ── Convergence traces via iteration_callback ──────────────────────────────
# Re-run one seed per function/algo to capture per-iteration traces.
TRACE_SEED = 0
traces: dict[tuple[str, str], list[float]] = {}

for algo_name, cfg_factory in CONFIGS.items():
    for fn_name, spec in BENCHMARKS.items():
        history: list[float] = []
        best_so_far: list[float] = [float("inf")]

        def _cb(iteration: int, cost: float, solution):
            if cost < best_so_far[0]:
                best_so_far[0] = cost
            history.append(best_so_far[0])

        cfg = cfg_factory(TRACE_SEED)
        givp(
            spec["func"],
            spec["bounds"],
            config=cfg,
            seed=TRACE_SEED,
            iteration_callback=_cb,
        )
        traces[(fn_name, algo_name)] = history

# Plot
fig, axes = plt.subplots(1, len(fn_names), figsize=(4 * len(fn_names), 4), sharey=False)

for ax, fn_name in zip(axes, fn_names):
    for algo_name in algo_names:
        trace = traces[(fn_name, algo_name)]
        ax.plot(
            range(1, len(trace) + 1),
            trace,
            label=algo_name,
            color=palette[algo_name],
            linewidth=1.8,
        )
    ax.set_title(fn_name, fontsize=12, fontweight="bold")
    ax.set_xlabel("Iteration", fontsize=9)
    ax.set_ylabel("Best objective (log)", fontsize=9)
    ax.set_yscale("symlog", linthresh=1e-6)
    ax.legend(fontsize=8)
    ax.grid(linestyle="--", alpha=0.4)

fig.suptitle(
    f"Convergence traces (seed={TRACE_SEED})",
    fontsize=13,
    fontweight="bold",
    y=1.01,
)
plt.tight_layout()
plt.savefig(
    "benchmark_convergence.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

## 6. Statistical significance (Wilcoxon signed-rank test)

We compare GIVP-full vs. GRASP-only on objective values across 30 runs using the
non-parametric Wilcoxon signed-rank test (α = 0.05).

In [ ]:
from scipy import stats  # noqa: E402  (requires scipy in notebooks extra)

print(f"{'Function':<14} {'W-statistic':>12} {'p-value':>12} {'Significant?':>13}")
print("-" * 55)
for fn_name in fn_names:
    a = df[(df["function"] == fn_name) & (df["algorithm"] == "GIVP-full")]["fun"].values
    b = df[(df["function"] == fn_name) & (df["algorithm"] == "GRASP-only")]["fun"].values
    stat, p = stats.wilcoxon(a, b, alternative="less")
    sig = "YES ✓" if p < 0.05 else "no"
    print(f"{fn_name:<14} {stat:>12.1f} {p:>12.4f} {sig:>13}")

## 7. Export results

In [ ]:
# ── Save raw records ───────────────────────────────────────────────────────
out_path = Path("benchmark_literature_comparison_results.json")
payload = {
    "metadata": {
        "n_dims": N_DIMS,
        "n_runs": N_RUNS,
        "seeds": SEEDS,
        "algorithms": list(CONFIGS),
        "benchmarks": list(BENCHMARKS),
    },
    "summary": json.loads(summary.to_json(orient="records")),
    "records": records,
}
out_path.write_text(json.dumps(payload, indent=2))
print(f"Results saved → {out_path.resolve()}")

# Also print the display table one more time for easy copy-paste into README
print("\n" + "=" * 90)
print("TABLE FOR README")
print("=" * 90)
print(disp.to_string(index=False))